In [1]:
import sys
import warnings

sys.path.append("/home/jovyan/projects/oslofjord_nutrient_trends/code")

import matplotlib.pyplot as plt
import nivapy3 as nivapy
import numpy as np
import pandas as pd
import seaborn as sn
import utils

warnings.simplefilter("ignore")
plt.style.use("ggplot")

In [2]:
COLOUR_DICT = {
    "Akvakultur": "royalblue",
    "Jordbruk": "sienna",
    "Avløp": "red",
    "Industri": "darkgrey",
    "Bebygd": "gold",
    "Bakgrunn": "limegreen",
}

# TEOTIL3 for Lillestrøm kommune
# Notebook 05: Comparision with TEOTIL

## 1. Sites of interest

**Note:** Sagelva, Nitelva and Leira are all reasonably large catchments comprising several regine units. This makes them suitable for modelling using TEOTIL3. Rømua and Åa are both single regine catchments, which is not ideal for TEOTIL3. Rømua is larger than Åa, so I would expect results for Rømua to be better than for Åa, but neither is really suitable for modelling using TEOTIL3.

In [3]:
# Read site data
xl_path = r"../data/lillestrom_monitoring_sites.xlsx"
stn_df = pd.read_excel(xl_path, sheet_name="chem_stns")

# Just the downstream stations close to regine outlets
dst_stns = [
    # Suitable for TEOTIL
    "002-29659",  # Leira
    "002-30586",  # Nitelva
    "002-46599",  # Sagelva
    # Single regine
    "002-28960",  # Rømua
    "002-60929",  # Åa
    # Part of regine
    "002-60933",  # Gansåa
    "002-30593",  # Jeksla
]
stn_df = stn_df.query("station_id in @dst_stns")

stn_df

,catchment,station_id,station_name,regine,wb_id,wb_name,lon,lat,comment
0,Sagelva,002-46599,Sagelva ved Skjetten bro (F3),002.CBA0,002-3899-R,Fjellhamarelva - Sagelva,11.01695,59.95850,Downstream
2,Nitelva,002-30586,Rud Nitelva (N8 ),002.CB0,002-3891-R,Nedre Nitelva,11.05700,59.94553,Downstream
4,Leira,002-29659,Leira ved Borgen bru (L5),002.CAA0,002-3384-R,Leira nedstrøms Krokfoss,11.10032,59.95036,Downstream
6,Rømua,002-28960,Rømua ved Lørenfallet - RØM1,002.D2Z,002-3659-R,Rømua,11.22099,60.02163,Downstream
7,Jeksla,002-30593,"Jeksla ved Haugli, J14",002.CAA0,002-599-R,Jeksla,11.10637,60.00199,Downstream. Small part of regine
8,Åa,002-60929,Fossåa ved Sylta (ÅA1),002.D3Z,002-3685-R,Fossåa,11.30018,59.99209,Downstream
11,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,002-3655-R,Gansåa,11.22275,59.86139,Downstream. Small part of regine


## 2. Read TEOTIL3 data

In [4]:
# TEOTIL options
st_yr = 2013
end_yr = 2023
nve_data_yr = 2024
agri_loss_model = "annual"
pars = ["TOTN", "TOTP", "TOC", "SS"]

In [5]:
# Aggregate data
teo_df = utils.get_teotil3_results(
    st_yr,
    end_yr,
    stn_df["regine"].tolist(),
    agri_loss_model,
    nve_data_yr,
)
id_cols = ["regine", "Parameter", "År"]
df_list = []
for par in pars:
    par_df = utils.aggregate_parameters(teo_df, par.lower())
    par_df["Parameter"] = par
    val_cols = [col for col in par_df.columns if col not in id_cols]
    par_df = par_df[id_cols + val_cols].sort_values(id_cols)
    df_list.append(par_df)

# Merge
mod_df = pd.concat(df_list, axis="rows")
mod_df = pd.merge(
    stn_df[["catchment", "station_id", "station_name", "regine"]],
    mod_df,
    how="right",
    on="regine",
)
mod_df["Akvakultur"] = mod_df["Akvakultur"].fillna(0)
mod_df.head()

,catchment,station_id,station_name,regine,Parameter,År,Jordbruk,Avløp,Industri,Bebygd,Bakgrunn,Akvakultur
0,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,TOTN,2013,6022.368720,1599.714993,26.585545,323.213056,4520.101501,0.0
1,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,TOTN,2014,6619.250937,1572.912456,27.364696,383.069527,5246.120463,0.0
2,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,TOTN,2015,5073.158833,1698.368498,26.929846,352.463377,4730.592281,0.0
3,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,TOTN,2016,5315.160030,1782.624309,33.665284,279.301439,3940.776391,0.0
4,Gansåa,002-60933,Gansåa nedstrøms Dalen RA (BD),002.C5,TOTN,2017,5267.650965,1741.250700,29.873686,320.746379,4391.917364,0.0


## 3. Read observed loads

From Notebook 04.

In [6]:
xl_path = r"../data/annual_loads.xlsx"
obs_df = pd.read_excel(xl_path, sheet_name="annual_loads")
obs_df.head()

,station_code,year,TOTN_tonn,TOTP_tonn,SS_tonn,TOC_tonn
0,002-28960,1990,96.300590,3.134720,1125.209562,374.573984
1,002-28960,1991,188.748221,8.012179,3578.643200,520.207794
2,002-28960,1992,170.753689,5.391133,3016.063269,459.973911
3,002-28960,1998,232.150551,8.431335,3146.030529,662.295678
4,002-28960,1999,234.147542,24.474299,19898.397088,NaN


## 4. Compare modelled to observed

In [7]:
cat_df = pd.read_excel(r"../data/vannmiljo_catch_metadata.xlsx")

val_cols = ["Jordbruk", "Avløp", "Industri", "Bebygd", "Bakgrunn"]
for idx, row in stn_df.iterrows():
    # Get modelled and observed data for station
    stn_id = row["station_id"]
    reg_id = row["regine"]
    name = row["catchment"]
    mod_stn_df = mod_df.query("station_id == @stn_id").copy()
    obs_stn_df = obs_df.query("(station_code == @stn_id) and (year >= 2013)").copy()

    # Scale to allow for regine outflows not exactly matching monitoring sites
    obs_area = cat_df.query("station_code == @stn_id")["cat_area_km2"].iloc[0]
    mod_area = teo_df.query("regine == @reg_id")["accum_upstr_area_km2"].iloc[0]
    mod_stn_df[val_cols] = mod_stn_df[val_cols] * obs_area / mod_area

    # Plot
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 6))
    axes = axes.flatten()
    handles, labels = None, None
    for ax_idx, par in enumerate(pars):
        ax = axes[ax_idx]
        mod_par_df = (
            mod_stn_df.query("Parameter == @par")[["År"] + val_cols]
            .sort_values("År")
            .set_index("År")
        )
        obs_par_df = (
            obs_stn_df[["year", f"{par}_tonn"]]
            .dropna(subset=f"{par}_tonn")
            .sort_values("year")
            .set_index("year")
        )

        # Stacked bar chart for modelled
        bottom = None
        for col in val_cols:
            values = mod_par_df[col]
            if bottom is None:
                ax.bar(
                    mod_par_df.index,
                    values,
                    label=col,
                    color=COLOUR_DICT[col],
                )
                bottom = values.copy()
            else:
                ax.bar(
                    mod_par_df.index,
                    values,
                    bottom=bottom,
                    label=col,
                    color=COLOUR_DICT[col],
                )
                bottom += values

        # Line chart for observed
        ax.plot(
            obs_par_df.index,
            obs_par_df[f"{par}_tonn"],
            color="red",
            marker="o",
            linestyle="-",
            label="Observed",
            linewidth=2,
        )

        ax.set_title(par)
        ax.set_ylabel("Tilførsler (tonn)")
        if handles is None:
            handles, labels = ax.get_legend_handles_labels()

    # Legend
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=3,
        frameon=False,
    )

    fig.suptitle(f"{name} ({stn_id})")
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])

    # Save
    png_path = f"../plots/mod_vs_obs_{name}_{stn_id}.png"
    plt.savefig(png_path, dpi=200, bbox_inches="tight")

    plt.close()